# Feature Engineering en SQL

A continuación, veremos cómo calcular diferentes variables para el feature engineering utilizando SQL.


In [ ]:
%pip install jupysql
%pip install duckdb-engine

In [ ]:
import duckdb
import pandas as pd

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

%sql duckdb://

In [ ]:
from pathlib import Path

project_path = Path(r'C:\Users\tomas\OneDrive\Documents\Maestria\DMEyF\dmeyf2026')
dataset_path = (project_path / 'data').as_posix() + '/'
dataset_file = 'competencia_01.csv'
csv_path = Path(dataset_path) / dataset_file
if not csv_path.is_file():
    raise FileNotFoundError(f'No se encontro el dataset: {csv_path}')
print(f'Dataset local: {csv_path}')


In [ ]:
%%sql
create or replace table competencia_01 as
select *
from read_csv_auto('{{dataset_path + dataset_file}}')


In [ ]:
%%sql
select
    Master_Fvencimiento
    , Visa_Fvencimiento
    , greatest(Master_Fvencimiento, Visa_Fvencimiento) as tc_fvencimiento_mayor
    , least(Master_Fvencimiento, Visa_Fvencimiento) as tc_fvencimiento_menor
from competencia_01 limit 10

Lo siguiente es querer operar dos variables, como por ejemplo sumarla. Esto es sencillo


In [ ]:
%%sql
select
    Master_msaldototal
    , Visa_msaldototal
    , Master_msaldototal + Visa_msaldototal as tc_saldo_total
from competencia_01 limit 10

Pero un DS de a de veras mirará los datos y se encontrará con un campo que es null cuando se lo suma a otro dará null.

In [ ]:
%%sql
select
    Master_msaldototal
    , Visa_msaldototal
    , Master_msaldototal + Visa_msaldototal as tc_saldo_total
from competencia_01 where Master_msaldototal is null limit 10

Esto no siempre es deseable y puede ser fácilmente evitable

In [ ]:
%%sql
select
    Master_msaldototal
    , Visa_msaldototal
    , ifnull(Master_msaldototal, 0) + ifnull(Visa_msaldototal, 0) as tc_saldo_total
from competencia_01 limit 10

In [ ]:
%%sql
CREATE OR REPLACE MACRO suma_sin_null(a, b) AS ifnull(a, 0) + ifnull(b, 0);


In [ ]:
%%sql
select distinct
    Master_msaldototal
    , Visa_msaldototal
    , suma_sin_null(Master_msaldototal, Visa_msaldototal) as tc_saldo_total
from competencia_01 where Master_msaldototal is null limit 10


TAREA: Escriba una macro para hacer un ratio de dos variables que sea seguro, donde no solo hay campos con null, también esta el problema de la división por cero. Como es costumbre comparta su solución por este canal. Lea https://duckdb.org/docs/sql/functions/numeric.html para referencias de funciones que puede usar.

---

"Claro!" me dirá, mientras lee esto con un mate en la mano, "para cosas fáciles usar SQL alcanza, pero para algo más complicado como crear campos contra el data drifting es difícil".... elija su medicina:

In [ ]:
%%sql
select
    foto_mes
    , numero_de_cliente
    , cliente_antiguedad
    , row_number() over (partition by numero_de_cliente order by foto_mes) as cliente_antiguedad_2
    , percent_rank() over (partition by foto_mes order by cliente_antiguedad) as cliente_antiguedad_3
    , cume_dist() over (partition by foto_mes order by cliente_antiguedad) as cliente_antiguedad_4
    , ntile(4) over (partition by foto_mes order by cliente_antiguedad) as cliente_antiguedad_5
    , ntile(10) over (partition by foto_mes order by cliente_antiguedad) as cliente_antiguedad_6
from competencia_01
order by numero_de_cliente, cliente_antiguedad


Qué paso? use las hermosas funciones analíticas de SQL. Al campo cliente_antiguedad (que no sufre de data drifting, solo esta para dar el ejemplo) para cada período (partition by foto_mes) la ordeno (order by cliente_antiguedad) y luego calculo las métricas de orden que pueden encontrar acá https://duckdb.org/docs/sql/window_functions.html#general-purpose-window-functions.

Seguiremos usando las funciones analíticas de SQL, esta vez para calcular features que utilizan valores del pasado.

Qué pasa si quiero agregar un feature que muestre el valor del periodo anterior?


In [ ]:
%%sql
select
  numero_de_cliente
  , foto_mes
  , ctrx_quarter
  , lag(ctrx_quarter, 1) over (partition by numero_de_cliente order by foto_mes) as lag_1_ctrx_quarter
from competencia_01
limit 10


Podemos calcular el delta (diferencia) entre el valor pasado y el presente, para uno o varios meses


In [ ]:
%%sql
select
  numero_de_cliente
  , foto_mes
  , ctrx_quarter
  , lag(ctrx_quarter, 1) over (partition by numero_de_cliente order by foto_mes) as lag_1_ctrx_quarter
  , ctrx_quarter - lag_1_ctrx_quarter as delta_1_ctrx_quarter
  , ctrx_quarter - lag(ctrx_quarter, 2) over (partition by numero_de_cliente order by foto_mes) as lag_2_ctrx_quarter
from competencia_01
limit 10


Si necesitamos ya no solo traer un valor del pasado, sino una secuencia de valores, por ejemplo para calcular la media móvil con los últimos 3 meses anteriores? se puede hacer fácilmente


In [ ]:
%%sql
select
  numero_de_cliente
  , foto_mes
  , ctrx_quarter
  , lag(ctrx_quarter, 1) over (partition by numero_de_cliente order by foto_mes) as lag_1_ctrx_quarter
  , lag(ctrx_quarter, 2) over (partition by numero_de_cliente order by foto_mes) as lag_2_ctrx_quarter
  , lag(ctrx_quarter, 3) over (partition by numero_de_cliente order by foto_mes) as lag_3_ctrx_quarter
  , avg(ctrx_quarter) over (partition by numero_de_cliente
                            order by foto_mes
                            rows between 3 preceding and current row) as avg_3_ctrx_quarter
from competencia_01
order by numero_de_cliente, foto_mes desc
limit 10


Si embargo puede resultar incómodo escribir constantemente el over partition sobre todo si se buscan aplicar muchas veces para distintas funciones. Para reducir el código se puede usar la siguiente sintaxis



In [ ]:
%%sql
select
  numero_de_cliente
  , foto_mes
  , ctrx_quarter
  , avg(ctrx_quarter) over ventana_3 as ctrx_quarter_media_3
  , max(ctrx_quarter) over ventana_3 as ctrx_quarter_max_3
  , min(ctrx_quarter) over ventana_3 as ctrx_quarter_min_3
from competencia_01
window ventana_3 as (partition by numero_de_cliente order by foto_mes rows between 3 preceding and current row)
limit 10


Para saber más que funciones tenemos disponibles, recomiendo ver los siguientes links:

https://duckdb.org/docs/archive/0.8.1/sql/window_functions
https://duckdb.org/docs/archive/0.8.1/sql/aggregates
Un caso más, que ni me voy a molestar en explicar que significa...


In [ ]:
%%sql
select
  numero_de_cliente
  , foto_mes
  , ctrx_quarter
  ,regr_slope(ctrx_quarter, cliente_antiguedad) over ventana_3 as ctrx_quarter_slope_3
from competencia_01
window ventana_3 as (partition by numero_de_cliente order by foto_mes rows between 3 preceding and current row)
limit 10


... Alguno dirá "tenemos que escribir todo esto a mano? Son muchas variables!". Bueno no, use los conocimientos de programación para que la computadora trabaje para usted. Si tenemos una lista de campos


In [ ]:
campos = ['active_quarter', 'cliente_vip', 'internet', 'cliente_edad', 'cliente_antiguedad', 'mrentabilidad']


Podemos hacer un script muy sencillo que nos genere el texto que hay que poner en una query para generar esas variables


In [ ]:
nuevos_features = ""
for campo in campos:
  nuevos_features += f"\n, regr_slope({campo}, cliente_antiguedad) over ventana_3 as ctrx_{campo}_slope_3"
print(nuevos_features)





Con la salida de esa celda, arme la query agregando las nuevas líneas y la ejecuta.

Lo que acabamos de hacer de manera muy simple es como "funcionan" sistemas como **dbt** que están tan de moda en el mundo de los datos.

La última reflexión, la creación de nuevas features es un proceso computacionalmente rápido pero intenso. Si ejecutó lo anterior pudo haber visto que en poco minutos tenía sus nuevas variables. Pero, también pudo haberle fallado por temas de recursos. Miles de variables necesitan los recursos adecuados. Use la nube, una máquina grande, al menos que sepa bien como optimizar las queries.


## Trabajo TH: actividad, productos y churn

Empezamos con una hipótesis acotada: la actividad es central para caracterizar la baja, pero su significado puede depender de cuántos productos tiene el cliente. Un cliente con varios productos y poca actividad puede estar reduciendo el uso efectivo del banco.

Para la descripción agrupamos `BAJA+1` y `BAJA+2` como `BAJA`, pero conservamos ambas clases separadas en los resultados. Para modelar y calcular ganancia, la clase accionable seguirá siendo `BAJA+2`. Usamos marzo-junio porque son los meses con objetivo completo.


In [ ]:
%%sql
select
    case when clase_ternaria = 'CONTINUA' then 'CONTINUA' else 'BAJA' end as clase
    , count(*) as cantidad
    , round(avg(ctrx_quarter), 2) as actividad_media
    , median(ctrx_quarter) as actividad_mediana
    , round(100.0 * avg((ctrx_quarter = 0)::int), 2) as actividad_cero_pct
    , median(cproductos) as productos_mediana
from competencia_01
where foto_mes between 202103 and 202106
group by clase
order by clase


En estos cuatro meses, la mediana de `ctrx_quarter` es 20 para las bajas y 106 para `CONTINUA`. Además, el 12,32% de las bajas tiene actividad cero, frente al 0,89% de quienes continúan. Es una asociación descriptiva: no demuestra causalidad ni garantiza mejora en un modelo.

Antes de construir el ratio, cruzamos bandas exploratorias de actividad y productos. Los cortes ayudan a leer la tabla; no se presentan como umbrales óptimos.


In [ ]:
%%sql
with perfil as (
    select
        clase_ternaria
        , case
            when ctrx_quarter = 0 then '0'
            when ctrx_quarter <= 30 then '1-30'
            when ctrx_quarter <= 60 then '31-60'
            when ctrx_quarter <= 120 then '61-120'
            else '121+'
          end as banda_actividad
        , case
            when cproductos <= 5 then '3-5'
            when cproductos <= 7 then '6-7'
            when cproductos <= 9 then '8-9'
            else '10+'
          end as banda_productos
    from competencia_01
    where foto_mes between 202103 and 202106
)
select
    banda_actividad
    , banda_productos
    , count(*) as cantidad
    , round(100.0 * count(*) filter (where clase_ternaria = 'BAJA+1') / count(*), 2) as baja_1_pct
    , round(100.0 * count(*) filter (where clase_ternaria = 'BAJA+2') / count(*), 2) as baja_2_pct
    , round(100.0 * count(*) filter (where clase_ternaria <> 'CONTINUA') / count(*), 2) as baja_total_pct
from perfil
group by banda_actividad, banda_productos
order by banda_actividad, banda_productos


El cruce sugiere que las dos dimensiones aportan información. Dentro de la banda de 1 a 30 transacciones, `BAJA+2` representa 5,72% entre quienes tienen 3 a 5 productos, 2,63% con 6 a 7, 2,04% con 8 a 9 y 1,14% con 10 o más. Esto motiva probar una medida de actividad relativa a la cantidad de productos.

La comparación mezcla observaciones de cuatro meses y una persona puede aparecer más de una vez. La usaremos para formular la hipótesis, no como prueba estadística independiente.


In [ ]:
%%sql
create or replace macro ratio_seguro(numerador, denominador) as
    case
        when numerador is null or denominador is null or denominador = 0 then null
        else numerador::double / denominador::double
    end;


In [ ]:
%%sql
select
    ratio_seguro(10, 2) as caso_valido
    , ratio_seguro(10, 0) as denominador_cero
    , ratio_seguro(null, 2) as numerador_nulo
    , ratio_seguro(10, null) as denominador_nulo


### Diversidad de canales

Como aproximación a la diversidad de comportamiento, contamos cuántos de ocho canales presentan uso: home banking, aplicación móvil, débito, Visa, Mastercard, ATM, cajas y call center. Las ocho columnas están completas en los seis meses. La macro conserva un nulo como desconocido; en los datos actuales no se produce ese caso.


In [ ]:
%%sql
create or replace macro indicador_uso(valor) as
    case
        when valor is null then null
        when valor > 0 then 1
        else 0
    end;


In [ ]:
%%sql
create or replace table competencia_01_fe_TH as
with intrames as (
    select
        *
        , ratio_seguro(ctrx_quarter, cproductos) as actividad_por_producto
        , indicador_uso(chomebanking_transacciones)
      + indicador_uso(cmobile_app_trx)
      + indicador_uso(ctarjeta_debito_transacciones)
      + indicador_uso(ctarjeta_visa_transacciones)
      + indicador_uso(ctarjeta_master_transacciones)
      + indicador_uso(catm_trx)
      + indicador_uso(ccajas_transacciones)
          + indicador_uso(ccallcenter_transacciones)
          as cantidad_canales_activos
    from competencia_01
), con_historia as (
    select
        actual.*
        , anterior.ctrx_quarter as lag_1_ctrx_quarter
        , anterior.cproductos as lag_1_cproductos
        , anterior.cantidad_canales_activos as lag_1_cantidad_canales_activos
        , anterior.numero_de_cliente is not null as historial_mes_anterior_disponible
    from intrames actual
    left join intrames anterior
      on anterior.numero_de_cliente = actual.numero_de_cliente
     and anterior.foto_mes // 100 * 12 + anterior.foto_mes % 100
         = actual.foto_mes // 100 * 12 + actual.foto_mes % 100 - 1
)
select
    *
    , ctrx_quarter - lag_1_ctrx_quarter as delta_1_ctrx_quarter
    , cproductos - lag_1_cproductos as delta_1_cproductos
    , cantidad_canales_activos - lag_1_cantidad_canales_activos
      as delta_1_cantidad_canales_activos
from con_historia;


In [ ]:
%%sql
select
    count(*) as filas
    , count(actividad_por_producto) as valores_validos
    , count(*) filter (where actividad_por_producto is null) as valores_nulos
    , count(*) filter (where not isfinite(actividad_por_producto)) as valores_no_finitos
    , count(*) filter (where cantidad_canales_activos is null) as canales_nulos
    , min(cantidad_canales_activos) as canales_minimo
    , max(cantidad_canales_activos) as canales_maximo
    , min(actividad_por_producto) as minimo
    , median(actividad_por_producto) as mediana
    , max(actividad_por_producto) as maximo
from competencia_01_fe_TH


In [ ]:
%%sql
select
    cantidad_canales_activos
    , count(*) as cantidad
    , round(100.0 * count(*) filter (where clase_ternaria = 'BAJA+1') / count(*), 2) as baja_1_pct
    , round(100.0 * count(*) filter (where clase_ternaria = 'BAJA+2') / count(*), 2) as baja_2_pct
    , round(100.0 * count(*) filter (where clase_ternaria <> 'CONTINUA') / count(*), 2) as baja_total_pct
from competencia_01_fe_TH
where foto_mes between 202103 and 202106
group by cantidad_canales_activos
order by cantidad_canales_activos


Las bajas utilizan una mediana de 2 canales, frente a 4 en `CONTINUA`. La tasa de `BAJA+2` cae de 5,82% sin canales activos a 3,32% con uno, 1,45% con dos, 0,65% con tres y 0,29% con cuatro. A partir de cinco canales la tasa se mantiene cerca de 0,1%; los grupos de siete y ocho canales son más chicos y no mantienen una caída estrictamente monótona.


### Primer bloque histórico: cambio mensual

Para actividad, productos y canales creamos el valor del mes calendario anterior y la diferencia contra el valor actual. El `left join` exige una distancia exacta de un mes; si un cliente salta un período, no hacemos pasar un registro más antiguo por el mes anterior. Tampoco usamos columnas del objetivo anterior ni información futura.


In [ ]:
%%sql
select
    foto_mes
    , count(*) as filas
    , count(*) filter (where historial_mes_anterior_disponible) as con_mes_anterior
    , count(*) filter (where not historial_mes_anterior_disponible) as sin_mes_anterior
    , round(100.0 * count(*) filter (where historial_mes_anterior_disponible) / count(*), 2) as cobertura_pct
from competencia_01_fe_TH
group by foto_mes
order by foto_mes


In [ ]:
%%sql
select
    clase_ternaria
    , count(*) as cantidad
    , median(delta_1_ctrx_quarter) as mediana_delta_actividad
    , median(delta_1_cproductos) as mediana_delta_productos
    , median(delta_1_cantidad_canales_activos) as mediana_delta_canales
from competencia_01_fe_TH
where foto_mes = 202105
group by clase_ternaria
order by clase_ternaria


In [ ]:
%%sql
select
    case when clase_ternaria = 'CONTINUA' then 'CONTINUA' else 'BAJA' end as clase
    , count(*) as cantidad
    , round(avg(actividad_por_producto), 2) as media
    , median(actividad_por_producto) as mediana
from competencia_01_fe_TH
where foto_mes between 202103 and 202106
group by clase
order by clase


En el EDA, `actividad_por_producto` tiene mediana 3 para las bajas y 14 para `CONTINUA`. Esto muestra separación descriptiva, pero todavía no sabemos si agrega información predictiva frente a `ctrx_quarter` y `cproductos` originales. Esa pregunta se responderá comparando el mismo Random Forest con y sin la nueva columna.

## Evaluación: Random Forest base

Antes de sumar más variables, medimos un baseline con las 152 variables predictoras originales. Se excluyen `numero_de_cliente`, `foto_mes` y `clase_ternaria`. Usamos cinco particiones estratificadas y pareadas de mayo: 70% para entrenamiento y 30% para validación. La ganancia de validación se divide por 0,30 para expresarla en la escala del mes completo.

La imputación se ajusta dentro del entrenamiento de cada partición. Los hiperparámetros quedan congelados para que las comparaciones posteriores cambien solamente el conjunto de variables. Junio se reserva para la evaluación temporal final y no se utiliza en esta etapa.


In [ ]:
import gc
from time import perf_counter
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedShuffleSplit

GANANCIA_ACIERTO = 1_072_500
COSTO_ESTIMULO = 27_500
UMBRAL = 0.025
PROP_VALIDACION = 0.30
SEMILLA_SPLITS = 1704
N_SPLITS = 5

PARAMETROS_RF = {
    'n_estimators': 200,
    'max_depth': 24,
    'min_samples_split': 3,
    'min_samples_leaf': 89,
    'max_features': 0.52,
    'max_samples': 0.70,
    'n_jobs': -1,
}

resultados_path = project_path / 'monday' / 'resultados_clase04'
resultados_path.mkdir(parents=True, exist_ok=True)


In [ ]:
datos_mayo = %sql select * from competencia_01_fe_TH where foto_mes = 202105

columnas_no_predictoras = ['numero_de_cliente', 'foto_mes', 'clase_ternaria']
features_intrames = ['actividad_por_producto', 'cantidad_canales_activos']
features_historicas = [
    'lag_1_ctrx_quarter', 'delta_1_ctrx_quarter',
    'lag_1_cproductos', 'delta_1_cproductos',
    'lag_1_cantidad_canales_activos', 'delta_1_cantidad_canales_activos',
    'historial_mes_anterior_disponible',
]
features_nuevas = features_intrames + features_historicas
columnas_base = [
    columna for columna in datos_mayo.columns
    if columna not in columnas_no_predictoras + features_nuevas
]

X_mayo = datos_mayo[columnas_base].astype('float32')
y_mayo = datos_mayo['clase_ternaria'].astype('string')

print(f'Mayo: {len(datos_mayo):,} filas')
print(f'Predictores originales: {len(columnas_base)}')
print(y_mayo.value_counts())


In [ ]:
def evaluar_random_forest(nombre, X, y, splits):
    filas = []
    for numero_split, (idx_train, idx_valid) in enumerate(splits):
        inicio = perf_counter()
        imputador = SimpleImputer(strategy='median')
        X_train = imputador.fit_transform(X.iloc[idx_train])
        X_valid = imputador.transform(X.iloc[idx_valid])

        modelo = RandomForestClassifier(
            **PARAMETROS_RF, random_state=SEMILLA_SPLITS + numero_split
        )
        modelo.fit(X_train, y.iloc[idx_train])

        indice_baja2 = list(modelo.classes_).index('BAJA+2')
        prob_baja2 = modelo.predict_proba(X_valid)[:, indice_baja2]
        estimulado = prob_baja2 >= UMBRAL
        real_baja2 = y.iloc[idx_valid].to_numpy() == 'BAJA+2'
        aciertos = int((estimulado & real_baja2).sum())
        errores = int((estimulado & ~real_baja2).sum())
        ganancia_muestra = aciertos * GANANCIA_ACIERTO - errores * COSTO_ESTIMULO

        filas.append({
            'modelo': nombre,
            'split': numero_split,
            'ganancia': ganancia_muestra / PROP_VALIDACION,
            'estimulados': int(estimulado.sum()),
            'aciertos_baja2': aciertos,
            'errores': errores,
            'baja2_validacion': int(real_baja2.sum()),
            'recall_baja2_pct': 100 * aciertos / real_baja2.sum(),
            'segundos': perf_counter() - inicio,
        })
        print(f'{nombre} · split {numero_split + 1}/{len(splits)} · ganancia {filas[-1]["ganancia"]:,.0f}')
        del modelo, imputador, X_train, X_valid, prob_baja2
        gc.collect()
    return pd.DataFrame(filas)


In [ ]:
sss_mayo = StratifiedShuffleSplit(
    n_splits=N_SPLITS, test_size=PROP_VALIDACION, random_state=SEMILLA_SPLITS
)
splits_mayo = list(sss_mayo.split(X_mayo, y_mayo))

resultados_baseline = evaluar_random_forest(
    'A · Originales', X_mayo, y_mayo, splits_mayo
)
resultados_baseline.to_csv(
    resultados_path / 'rf_baseline_mayo.csv', index=False
)
display(resultados_baseline)
display(resultados_baseline[['ganancia', 'estimulados', 'aciertos_baja2',
                             'errores', 'recall_baja2_pct', 'segundos']].agg(['mean', 'std']))


### Primera comparación: actividad por producto

Agregamos únicamente `actividad_por_producto`. Las particiones y las semillas de los árboles son las mismas que en el baseline. Así podemos calcular la diferencia de ganancia dentro de cada split.


In [ ]:
columnas_fe1 = columnas_base + ['actividad_por_producto']
X_mayo_fe1 = datos_mayo[columnas_fe1].astype('float32')

resultados_fe1 = evaluar_random_forest(
    'B · + actividad_por_producto', X_mayo_fe1, y_mayo, splits_mayo
)
resultados_fe1.to_csv(
    resultados_path / 'rf_fe1_actividad_por_producto_mayo.csv', index=False
)

comparacion_fe1 = (
    resultados_baseline[['split', 'ganancia']]
    .rename(columns={'ganancia': 'ganancia_base'})
    .merge(
        resultados_fe1[['split', 'ganancia']].rename(columns={'ganancia': 'ganancia_fe1'}),
        on='split', validate='one_to_one'
    )
)
comparacion_fe1['diferencia_fe1_menos_base'] = (
    comparacion_fe1['ganancia_fe1'] - comparacion_fe1['ganancia_base']
)
comparacion_fe1.to_csv(resultados_path / 'comparacion_fe1_mayo.csv', index=False)
display(comparacion_fe1.style.format('{:,.0f}', subset=[
    'ganancia_base', 'ganancia_fe1', 'diferencia_fe1_menos_base'
]))
print(f'Diferencia media FE1 - base: {comparacion_fe1["diferencia_fe1_menos_base"].mean():,.0f}')
print(f'FE1 gana en {(comparacion_fe1["diferencia_fe1_menos_base"] > 0).mean():.0%} de los splits')


### Resultado verificado de la primera comparación

El baseline obtuvo una ganancia media de **$254.778.333**, con un desvío de **$12.524.934**. Al agregar `actividad_por_producto`, la media bajó a **$250.983.333** y el desvío subió a **$15.096.362**. La diferencia pareada promedio fue **-$3.795.000** y la variante con FE ganó solamente en 1 de los 5 splits. También bajó el recall medio de `BAJA+2`, de 49,66% a 48,89%.

Por ahora, el ratio describe bien a las clases pero **no aportó una mejora predictiva incremental** al Random Forest, que ya dispone de `ctrx_quarter` y `cproductos`. Cinco particiones son pocas para una conclusión definitiva, pero el resultado no justifica ampliar todavía esta prueba. Conservamos la función y la variable, y avanzamos con una hipótesis distinta: diversidad de canales activos.

Este resultado será parte de la evidencia acumulada del feature engineering. No se evaluó junio y no se ajustaron parámetros después de observar la comparación.

### Segunda comparación: cantidad de canales activos

Como el ratio anterior no mejoró el baseline, no lo arrastramos automáticamente. El experimento C vuelve a partir de las variables originales y agrega solamente `cantidad_canales_activos`. Esto permite aislar la hipótesis de diversidad de uso.


In [ ]:
columnas_fe2 = columnas_base + ['cantidad_canales_activos']
X_mayo_fe2 = datos_mayo[columnas_fe2].astype('float32')

resultados_fe2 = evaluar_random_forest(
    'C · + cantidad_canales_activos', X_mayo_fe2, y_mayo, splits_mayo
)
resultados_fe2.to_csv(
    resultados_path / 'rf_fe2_canales_activos_mayo.csv', index=False
)

comparacion_fe2 = (
    resultados_baseline[['split', 'ganancia']]
    .rename(columns={'ganancia': 'ganancia_base'})
    .merge(
        resultados_fe2[['split', 'ganancia']].rename(columns={'ganancia': 'ganancia_fe2'}),
        on='split', validate='one_to_one'
    )
)
comparacion_fe2['diferencia_fe2_menos_base'] = (
    comparacion_fe2['ganancia_fe2'] - comparacion_fe2['ganancia_base']
)
comparacion_fe2.to_csv(resultados_path / 'comparacion_fe2_mayo.csv', index=False)
display(comparacion_fe2.style.format('{:,.0f}', subset=[
    'ganancia_base', 'ganancia_fe2', 'diferencia_fe2_menos_base'
]))
print(f'Diferencia media FE2 - base: {comparacion_fe2["diferencia_fe2_menos_base"].mean():,.0f}')
print(f'FE2 gana en {(comparacion_fe2["diferencia_fe2_menos_base"] > 0).mean():.0%} de los splits')


### Resultado verificado de la segunda comparación

El modelo con `cantidad_canales_activos` obtuvo una ganancia media de **$251.496.667**, con un desvío de **$19.979.820**. La diferencia pareada media contra el baseline fue **-$3.281.667** y ganó solamente en 1 de los 5 splits. Su recall medio de `BAJA+2` fue 49,20%, frente a 49,66% del baseline. La diversidad de canales caracteriza las clases, pero este agregado intrames tampoco añadió una mejora estable sobre sus componentes originales.

### Tercera comparación: cambios respecto del mes anterior

El experimento D agrega al baseline los lags y deltas de actividad, productos y canales, más un indicador de disponibilidad del mes anterior. No incluye los dos agregados intrames evaluados anteriormente salvo cuando son necesarios para describir la historia de canales.


In [ ]:
columnas_fe3 = columnas_base + features_historicas
X_mayo_fe3 = datos_mayo[columnas_fe3].astype('float32')

resultados_fe3 = evaluar_random_forest(
    'D · + historia de actividad', X_mayo_fe3, y_mayo, splits_mayo
)
resultados_fe3.to_csv(
    resultados_path / 'rf_fe3_historia_actividad_mayo.csv', index=False
)

comparacion_fe3 = (
    resultados_baseline[['split', 'ganancia']]
    .rename(columns={'ganancia': 'ganancia_base'})
    .merge(
        resultados_fe3[['split', 'ganancia']].rename(columns={'ganancia': 'ganancia_fe3'}),
        on='split', validate='one_to_one'
    )
)
comparacion_fe3['diferencia_fe3_menos_base'] = (
    comparacion_fe3['ganancia_fe3'] - comparacion_fe3['ganancia_base']
)
comparacion_fe3.to_csv(resultados_path / 'comparacion_fe3_mayo.csv', index=False)
display(comparacion_fe3.style.format('{:,.0f}', subset=[
    'ganancia_base', 'ganancia_fe3', 'diferencia_fe3_menos_base'
]))
print(f'Diferencia media FE3 - base: {comparacion_fe3["diferencia_fe3_menos_base"].mean():,.0f}')
print(f'FE3 gana en {(comparacion_fe3["diferencia_fe3_menos_base"] > 0).mean():.0%} de los splits')


### Resultado inicial del bloque histórico

El modelo histórico obtuvo una ganancia media de **$261.140.000**, $6.361.667 por encima del baseline, y ganó en 3 de los 5 splits. También aumentó el recall medio de `BAJA+2` de 49,66% a 50,73%. Sin embargo, su desvío fue **$33.776.424** y las diferencias pareadas abarcaron desde -$34.650.000 hasta +$44.366.667. Es el primer bloque con mejora promedio, pero la evidencia todavía es inestable. Antes de usar junio o afirmar que mejora, debemos ampliar la validación sobre mayo.

### Resumen de los experimentos intrames e históricos


In [ ]:
resultados_todos = pd.concat([
    resultados_baseline, resultados_fe1, resultados_fe2, resultados_fe3
], ignore_index=True)
resumen_experimentos = (
    resultados_todos.groupby('modelo')
    .agg(
        ganancia_media=('ganancia', 'mean'),
        ganancia_desvio=('ganancia', 'std'),
        estimulados_media=('estimulados', 'mean'),
        recall_baja2_medio=('recall_baja2_pct', 'mean'),
    )
    .reset_index()
)
resumen_experimentos.to_csv(
    resultados_path / 'resumen_experimentos_mayo.csv', index=False
)
display(resumen_experimentos.style.format({
    'ganancia_media': '{:,.0f}',
    'ganancia_desvio': '{:,.0f}',
    'estimulados_media': '{:,.1f}',
    'recall_baja2_medio': '{:.2f}%',
}))


Los dos agregados intrames caracterizan a las clases, pero no mejoran el RF sobre sus componentes originales. Los cambios mensuales sí muestran una mejora promedio inicial, acompañada por mucha variabilidad. Conservamos todos los generadores para documentar resultados positivos y negativos; la selección final se hará después de una validación más extensa y sin consultar junio.

### Exportación opcional

La siguiente celda escribe un CSV nuevo y conserva el archivo original. Ejecutarla solamente cuando queramos materializar la tabla ampliada.


In [ ]:
%%sql
COPY competencia_01_fe_TH TO '{{dataset_path}}competencia_01_fe_TH.csv' (FORMAT CSV, HEADER TRUE);
